<a href="https://colab.research.google.com/github/Md-Golam-Raiyhan/INSE-6290-/blob/main/data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

# --------------------------------------------
# Configuration from the notebook
# --------------------------------------------
FRESH_CATEGORIES = {"Salad", "Soup", "Fish", "Seafood"}

SHELF_LIFE_WEEKS = 1   # fresh items
SHELF_LIFE_SHELF = 2   # longer-life items

# --------------------------------------------
# 1. Load and aggregate real dataset
# --------------------------------------------
def load_real_data(top_n_meals: int = 10) -> pd.DataFrame:
    train = pd.read_csv(PROJECT_ROOT / "train.csv")
    meals = pd.read_csv(PROJECT_ROOT / "meal_info.csv")

    # Merge meal information
    df = train.merge(meals, on="meal_id", how="left")

    # Select top-N meals by total historical demand
    top_meals = (
        df.groupby("meal_id")["num_orders"]
        .sum()
        .nlargest(top_n_meals)
        .index
        .tolist()
    )

    df = df[df["meal_id"].isin(top_meals)].copy()

    # Aggregate across all fulfillment centers per (meal_id, week)
    agg = (
        df.groupby(["meal_id", "week"])
        .agg(
            num_orders=("num_orders", "sum"),
            checkout_price=("checkout_price", "mean"),
            base_price=("base_price", "mean"),
            emailer_for_promotion=("emailer_for_promotion", "max"),
            homepage_featured=("homepage_featured", "max"),
            category=("category", "first"),
            cuisine=("cuisine", "first"),
        )
        .reset_index()
    )

    # Derived price features
    agg["price_ratio"] = (agg["checkout_price"] / agg["base_price"]).round(4)
    agg["price_discount"] = (agg["base_price"] - agg["checkout_price"]).round(2)

    # Derived cost parameters
    agg["selling_price"] = agg["checkout_price"].round(2)
    agg["unit_cost"] = (agg["checkout_price"] * 0.40).round(2)
    agg["holding_cost"] = (agg["unit_cost"] * 0.05).round(3)
    agg["shortage_cost"] = (agg["checkout_price"] * 0.50).round(2)

    # Shelf life by category
    agg["shelf_life_weeks"] = agg["category"].apply(
        lambda c: SHELF_LIFE_WEEKS if c in FRESH_CATEGORIES else SHELF_LIFE_SHELF
    )

    # Seasonality feature
    agg["week_of_year"] = ((agg["week"] - 1) % 52) + 1

    return agg.sort_values(["meal_id", "week"]).reset_index(drop=True)

# --------------------------------------------
# 2. Feature engineering
# --------------------------------------------
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["meal_id", "week"]).copy()

    grp = df.groupby("meal_id")

    df["lag_1"] = grp["num_orders"].shift(1)
    df["lag_4"] = grp["num_orders"].shift(4)

    df["rolling_4"] = (
        grp["num_orders"].shift(1).rolling(4).mean().reset_index(level=0, drop=True)
    )

    df["rolling_8"] = (
        grp["num_orders"].shift(1).rolling(8).mean().reset_index(level=0, drop=True)
    )

    return df

# --------------------------------------------
# 3. Generate working datasets
# --------------------------------------------
# Load aggregated real dataset
agg_df = load_real_data(top_n_meals=10)

# Save aggregated dataset
agg_df.to_csv("aggregated_top10_meals_weekly.csv", index=False)

# Prepare features
prepared_df = prepare_features(agg_df).dropna().copy()

# Save prepared feature dataset
prepared_df.to_csv("prepared_dataset_with_features.csv", index=False)

# --------------------------------------------
# 4. Train-test split (same logic as notebook)
# --------------------------------------------
test_weeks = 10
cutoff = prepared_df["week"].max() - test_weeks + 1

train_prepared = prepared_df[prepared_df["week"] < cutoff].copy()
test_prepared = prepared_df[prepared_df["week"] >= cutoff].copy()

train_prepared.to_csv("train_prepared_dataset.csv", index=False)
test_prepared.to_csv("test_prepared_dataset.csv", index=False)

# --------------------------------------------
# 5. Save top-10 meal table too
# --------------------------------------------
train_raw = pd.read_csv("train.csv")
meal_info = pd.read_csv("meal_info.csv")

top_meals_table = (
    train_raw.groupby("meal_id", as_index=False)["num_orders"]
    .sum()
    .sort_values("num_orders", ascending=False)
    .head(10)
    .merge(meal_info, on="meal_id", how="left")
)

top_meals_table.to_csv("top_10_meals_by_total_demand.csv", index=False)

# --------------------------------------------
# 6. Print confirmation
# --------------------------------------------
print("Done.")
print("Aggregated dataset shape:", agg_df.shape)
print("Prepared dataset shape:", prepared_df.shape)
print("Train dataset shape:", train_prepared.shape)
print("Test dataset shape:", test_prepared.shape)
print("\nGenerated files:")
print("- top_10_meals_by_total_demand.csv")
print("- aggregated_top10_meals_weekly.csv")
print("- prepared_dataset_with_features.csv")
print("- train_prepared_dataset.csv")
print("- test_prepared_dataset.csv")

Done.
Aggregated dataset shape: (1450, 17)
Prepared dataset shape: (1370, 21)
Train dataset shape: (1270, 21)
Test dataset shape: (100, 21)

Generated files:
- top_10_meals_by_total_demand.csv
- aggregated_top10_meals_weekly.csv
- prepared_dataset_with_features.csv
- train_prepared_dataset.csv
- test_prepared_dataset.csv
